In [ ]:
# Climate_Change_Analysis.ipynb

# Imports Libs
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import Data
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
file_name = "Messstationen Monatsdaten v2 Datensatz_198501_202508.csv"
relative_path = os.path.join("..", "data", file_name)
csv_path = os.path.abspath(relative_path)

df = pd.read_csv(csv_path, parse_dates=["time"])

# Step 3: Preprocess
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month

# Check for missing values
print(df.isnull().sum())
display(df.head(5))
display(df.tail(5))


In [ ]:
# Annual Averages
# annual_avg = df.groupby("year")[["tl_i_mittel", "tlmin", "tlmax", "absf_mittel", "p", "so_h", "tage_schreg"]].mean().reset_index()

annual_avg = df.groupby("year")[["tl_i_mittel", "tlmin", "tlmax", "absf_mittel", "p", "so_h"]].mean()
annual_avg["tage_schreg"] = df.groupby("year")["tage_schreg"].sum()
annual_avg = annual_avg.reset_index()

# Plot Temperature Trends
plt.figure(figsize=(12, 6))
# sns.lineplot(data=annual_avg, x="year", y="tl_i_mittel", label="Avg Temp (°C)")
sns.lineplot(data=annual_avg, x="year", y="tlmax", label="Max Temp (°C)")
# sns.lineplot(data=annual_avg, x="year", y="tlmin", label="Min Temp (°C)")
plt.title("Temperature Trends Over Time")
plt.ylabel("Temperature (°C)")
plt.grid(True)
plt.legend()
plt.show()

# Sunshine Duration Trend
plt.figure(figsize=(12, 6))
sns.lineplot(data=annual_avg, x="year", y="so_h", label="Sunshine Duration (h)")
plt.title("Annual Sunshine Duration")
plt.ylabel("Hours")
plt.grid(True)
plt.show()

# Snow+Rain Days Trend
snow_rain = df.groupby("year")["tage_schreg"].sum().reset_index()
plt.figure(figsize=(12, 6))
sns.lineplot(data=snow_rain, x="year", y="tage_schreg", label="Snow+Rain Days")
plt.title("Annual Snow+Rain Days")
plt.ylabel("Days")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Linear Trend Analysis (Temperature)
X = annual_avg["year"].values.reshape(-1, 1)
y = annual_avg["tl_i_mittel"].values
model_temp = LinearRegression().fit(X, y)
trend_temp = model_temp.predict(X)

plt.figure(figsize=(12, 6))
plt.plot(annual_avg["year"], y, label="Avg Temp")
plt.plot(annual_avg["year"], trend_temp, label="Linear Trend", linestyle="--")
plt.title("Linear Trend in Avg Temperature")
plt.legend()
plt.grid(True)
plt.show()

# Linear Trend Analysis (Snow and Rain)
X = annual_avg["year"].values.reshape(-1, 1)
y = annual_avg["tage_schreg"].values
model_sr = LinearRegression().fit(X, y)
trend_sr = model_sr.predict(X)

plt.figure(figsize=(12, 6))
plt.plot(annual_avg["year"], y, label="Snow / Rain Days")
plt.plot(annual_avg["year"], trend_sr, label="Linear Trend", linestyle="--")
plt.title("Linear Trend in Snow / Rain Days")
plt.legend()
plt.grid(True)
plt.show()

# Correlation Matrix
plt.figure(figsize=(10, 6))
sns.heatmap(df[["tl_i_mittel", "tlmin", "tlmax", "absf_mittel", "p", "so_h", "tage_schreg"]].corr(), annot=True, cmap="coolwarm")
plt.title("Climate Variable Correlations")
plt.show()